# 04 Model Explainability

This version assumes the test split produced by Notebook 01, containing the six restricted breeds and the fixed sample of 24 unrestricted breeds. It verifies the established confusion matrix, then saves a compact reproducible sample of three TP, three TN, three FP, and three FN cases.


In [ ]:
# 1. Mount Drive and define paths
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/restricted-dog-cnn")
TEST_DIR = PROJECT_ROOT / "data" / "test"

FINE_TUNED_MODEL_PATH = (
    PROJECT_ROOT / "fine_tuning" / "checkpoints"
    / "FineTuned_InceptionResNetV2_best.keras"
)
FROZEN_MODEL_PATH = (
    PROJECT_ROOT / "model_zoo" / "checkpoints"
    / "InceptionResNetV2_best.keras"
)

OUTPUT_ROOT = PROJECT_ROOT / "model_explainability"
FIGURES_DIR = OUTPUT_ROOT / "figures"
STATUS_LOG = OUTPUT_ROOT / "status_log.txt"
RUN_STARTED_FILE = OUTPUT_ROOT / "RUN_STARTED.txt"
RUN_COMPLETED_FILE = OUTPUT_ROOT / "RUN_COMPLETED.txt"

print("Project root:", PROJECT_ROOT)
print("Test directory:", TEST_DIR)
print("Model explainability output:", OUTPUT_ROOT)


Mounted at /content/drive
Project root: /content/drive/MyDrive/restricted-dog-cnn
Test directory: /content/drive/MyDrive/restricted-dog-cnn/data/test
Version 10 output: /content/drive/MyDrive/restricted-dog-cnn/Explanation-v10


In [ ]:
# 2. Imports and settings
import gc
import random
import re
import shutil
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 58
IMAGE_SIZE = (299, 299)
CLASS_NAMES = ["unrestricted", "restricted"]
OVERLAY_ALPHA = 0.42
FLAT_TOLERANCE = 1e-8
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

EXPECTED_COUNTS = {"TP": 183, "TN": 824, "FP": 14, "FN": 7}
SAVE_COUNTS = {"TP": 3, "TN": 3, "FP": 3, "FN": 3}

PREPROCESSING_MODES = [
    "raw_0_255",
    "divide_255",
    "inception_resnet_v2",
]

RESET_OUTPUT = True
PREDICTION_BATCH_SIZE = 32
FILE_WAIT_ATTEMPTS = 20
FILE_WAIT_SECONDS = 0.25

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
plt.ioff()

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


TensorFlow: 2.20.0
GPU: []


In [ ]:
# 3. Create a clean model explainability output folder
if RESET_OUTPUT and OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RUN_STARTED_FILE.write_text(
    "Model explainability run started. If this file remains without "
    "RUN_COMPLETED.txt, the run did not finish.\n",
    encoding="utf-8",
)
STATUS_LOG.write_text(
    "Model explainability status log\n",
    encoding="utf-8",
)

write_test = OUTPUT_ROOT / "_write_test.txt"
write_test.write_text("Model explainability write test", encoding="utf-8")

if not write_test.exists() or write_test.stat().st_size == 0:
    raise RuntimeError(f"Cannot write to output folder: {OUTPUT_ROOT}")

write_test.unlink()
print("Drive write test passed:", OUTPUT_ROOT)


Drive write test passed: /content/drive/MyDrive/restricted-dog-cnn/Explanation-v10


In [ ]:
# 4. Validate data and choose the trained checkpoint
for class_name in CLASS_NAMES:
    class_dir = TEST_DIR / class_name
    if not class_dir.exists():
        raise FileNotFoundError(f"Missing test class folder: {class_dir}")

if FINE_TUNED_MODEL_PATH.exists():
    MODEL_PATH = FINE_TUNED_MODEL_PATH
    MODEL_STAGE = "fine_tuned"
elif FROZEN_MODEL_PATH.exists():
    MODEL_PATH = FROZEN_MODEL_PATH
    MODEL_STAGE = "frozen"
else:
    raise FileNotFoundError("No trained checkpoint was found.")

print("Model stage:", MODEL_STAGE)
print("Checkpoint:", MODEL_PATH)


Model stage: fine_tuned
Checkpoint: /content/drive/MyDrive/restricted-dog-cnn/fine_tuning/checkpoints/FineTuned_InceptionResNetV2_best.keras


In [ ]:
# 5. Load the model and construct a connected Grad-CAM graph
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

spatial_models = []
for layer in model.layers:
    if isinstance(layer, tf.keras.Model):
        try:
            if len(layer.output.shape) == 4:
                spatial_models.append(layer)
        except Exception:
            pass

if not spatial_models:
    raise RuntimeError("No nested CNN with a 4D spatial output was found.")

base_model = max(
    spatial_models,
    key=lambda layer: (int(layer.output.shape[-1] or 0), len(layer.layers)),
)

dense_layers = [
    layer for layer in model.layers
    if isinstance(layer, tf.keras.layers.Dense)
]
if not dense_layers:
    raise RuntimeError("No Dense classification layer was found.")

classification_layer = dense_layers[-1]
if classification_layer.units != 1:
    raise RuntimeError("Expected a one-unit binary classifier.")

# Re-run the top-level model graph from its input and capture the nested
# backbone output as it is actually used by the outer model. Referencing
# base_model.output directly can produce a disconnected Keras 3 graph.
x = model.input
feature_maps = None
restricted_logit = None
probability = None

for layer in model.layers:
    if isinstance(layer, tf.keras.layers.InputLayer):
        continue

    if layer is classification_layer:
        dense_input = x

        def compute_restricted_logit(tensor):
            value = tf.linalg.matmul(tensor, classification_layer.kernel)
            if classification_layer.use_bias:
                value = tf.nn.bias_add(value, classification_layer.bias)
            return value

        restricted_logit = tf.keras.layers.Lambda(
            compute_restricted_logit,
            name="restricted_logit",
        )(dense_input)

        probability = classification_layer(dense_input)
        x = probability
    else:
        x = layer(x)
        if layer is base_model:
            feature_maps = x

if feature_maps is None:
    raise RuntimeError("The backbone was not encountered in the model graph.")
if restricted_logit is None or probability is None:
    raise RuntimeError("The classification head could not be reconstructed.")

grad_model = tf.keras.Model(
    inputs=model.input,
    outputs=[feature_maps, restricted_logit, probability],
    name="connected_gradcam_model",
)

# Immediate smoke test: fail here rather than after processing the dataset.
dummy = tf.zeros((1, *IMAGE_SIZE, 3), dtype=tf.float32)
test_feature_maps, test_logit, test_probability = grad_model(
    dummy, training=False
)

print("Backbone:", base_model.name)
print("Spatial output:", test_feature_maps.shape)
print("Classifier:", classification_layer.name)
print("Grad-CAM graph smoke test passed.")


Backbone: inception_resnet_v2
Spatial output: (1, 8, 8, 1536)
Classifier: restricted_probability
Grad-CAM graph smoke test passed.


In [ ]:
# 6. Enumerate the complete test split
all_test_images = sorted(
    path
    for class_name in CLASS_NAMES
    for path in (TEST_DIR / class_name).rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

if not all_test_images:
    raise FileNotFoundError("No test images were found.")

true_classes = [path.parent.name for path in all_test_images]

print("Total images:", len(all_test_images))
print("Unrestricted:", true_classes.count("unrestricted"))
print("Restricted:", true_classes.count("restricted"))

if len(all_test_images) != sum(EXPECTED_COUNTS.values()):
    raise RuntimeError(
        f"Expected {sum(EXPECTED_COUNTS.values())} test images, "
        f"found {len(all_test_images)}."
    )


Total images: 1028
Unrestricted: 838
Restricted: 190


In [ ]:
# 7. Loading, prediction, selection, and Grad-CAM helpers
def log_status(message):
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {message}"
    print(line)
    with STATUS_LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")


def wait_for_file(path, min_bytes=1000):
    for _ in range(FILE_WAIT_ATTEMPTS):
        if path.exists() and path.stat().st_size >= min_bytes:
            return True
        time.sleep(FILE_WAIT_SECONDS)
    return False


def load_rgb(image_path):
    image = tf.keras.utils.load_img(
        image_path, target_size=IMAGE_SIZE, color_mode="rgb"
    )
    return tf.keras.utils.img_to_array(image)


def preprocess_tensor(image_tensor, mode):
    x = tf.cast(image_tensor, tf.float32)

    if mode == "raw_0_255":
        return x
    if mode == "divide_255":
        return x / 255.0
    if mode == "inception_resnet_v2":
        return tf.keras.applications.inception_resnet_v2.preprocess_input(x)

    raise ValueError(f"Unknown preprocessing mode: {mode}")


def make_prediction_dataset(paths, mode):
    path_strings = [str(path) for path in paths]
    ds = tf.data.Dataset.from_tensor_slices(path_strings)

    def decode(path):
        data = tf.io.read_file(path)
        image = tf.io.decode_image(
            data, channels=3, expand_animations=False
        )
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMAGE_SIZE, method="bilinear")
        return preprocess_tensor(image, mode)

    return (
        ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(PREDICTION_BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )


def confusion_counts(probabilities, threshold, positive_when_high=True):
    probabilities = np.asarray(probabilities)

    if positive_when_high:
        predicted_restricted = probabilities >= threshold
    else:
        predicted_restricted = probabilities < threshold

    true_restricted = np.array(
        [name == "restricted" for name in true_classes]
    )

    return {
        "TP": int(np.sum(true_restricted & predicted_restricted)),
        "TN": int(np.sum(~true_restricted & ~predicted_restricted)),
        "FP": int(np.sum(~true_restricted & predicted_restricted)),
        "FN": int(np.sum(true_restricted & ~predicted_restricted)),
    }


def find_exact_operating_point(probabilities):
    values = np.unique(np.asarray(probabilities, dtype=float))
    thresholds = np.concatenate((
        [np.nextafter(values.min(), -np.inf)],
        (values[:-1] + values[1:]) / 2.0,
        [np.nextafter(values.max(), np.inf)],
        [0.5],
    ))

    matches = []
    best = None

    for positive_when_high in [True, False]:
        for threshold in thresholds:
            counts = confusion_counts(
                probabilities, threshold, positive_when_high
            )
            distance = sum(
                abs(counts[key] - EXPECTED_COUNTS[key])
                for key in EXPECTED_COUNTS
            )

            candidate = {
                "threshold": float(threshold),
                "positive_when_high": positive_when_high,
                "distance": int(distance),
                **counts,
            }

            if best is None or candidate["distance"] < best["distance"]:
                best = candidate

            if counts == EXPECTED_COUNTS:
                matches.append(candidate)

    return matches, best


def classify_probability(probability, threshold, positive_when_high):
    if positive_when_high:
        restricted = probability >= threshold
    else:
        restricted = probability < threshold
    return "restricted" if restricted else "unrestricted"


def category_code(true_class, predicted_class):
    if true_class == "restricted" and predicted_class == "restricted":
        return "TP"
    if true_class == "unrestricted" and predicted_class == "unrestricted":
        return "TN"
    if true_class == "unrestricted" and predicted_class == "restricted":
        return "FP"
    return "FN"


def safe_stem(path):
    clean = re.sub(r"[^A-Za-z0-9._-]+", "_", path.stem)
    return clean.strip("._") or "image"


def normalise_map(raw_map):
    raw_map = tf.cast(raw_map, tf.float32)
    raw_map = tf.where(
        tf.math.is_finite(raw_map), raw_map, tf.zeros_like(raw_map)
    )
    minimum = tf.reduce_min(raw_map)
    maximum = tf.reduce_max(raw_map)
    spread = maximum - minimum

    if float(spread) <= FLAT_TOLERANCE:
        return raw_map.numpy(), False

    return ((raw_map - minimum) / spread).numpy(), True


def predict_and_explain(image_batch, target_class):
    """
    Generate Grad-CAM for the class already assigned during the verified
    full-dataset prediction pass.
    """
    if target_class not in CLASS_NAMES:
        raise ValueError(f"Unknown target class: {target_class}")

    with tf.GradientTape() as tape:
        feature_maps, logit, probability = grad_model(
            image_batch, training=False
        )

        restricted_score = logit[:, 0]

        if target_class == "restricted":
            target_score = (
                restricted_score
                if POSITIVE_WHEN_HIGH
                else -restricted_score
            )
        else:
            target_score = (
                -restricted_score
                if POSITIVE_WHEN_HIGH
                else restricted_score
            )

    gradients = tape.gradient(target_score, feature_maps)
    if gradients is None:
        raise RuntimeError("Gradients are None.")

    gradients = tf.where(
        tf.math.is_finite(gradients),
        gradients,
        tf.zeros_like(gradients),
    )
    feature_maps = tf.where(
        tf.math.is_finite(feature_maps),
        feature_maps,
        tf.zeros_like(feature_maps),
    )

    weights = tf.reduce_mean(gradients, axis=(1, 2))
    signed_cam = tf.reduce_sum(
        feature_maps * weights[:, None, None, :], axis=-1
    )[0]

    heatmap, valid = normalise_map(tf.nn.relu(signed_cam))
    method = "standard_positive_gradcam"

    if not valid:
        heatmap, valid = normalise_map(tf.abs(signed_cam))
        method = "absolute_signed_evidence_fallback"

    if not valid:
        contribution_map = tf.reduce_sum(
            tf.abs(feature_maps[0] * weights[0][None, None, :]),
            axis=-1,
        )
        heatmap, valid = normalise_map(contribution_map)
        method = "absolute_channel_contribution_fallback"

    if not valid or float(np.std(heatmap)) <= FLAT_TOLERANCE:
        raise RuntimeError("Heatmap remained flat.")

    return heatmap, {
        "target_class": target_class,
        "single_image_model_output": float(
            probability.numpy().reshape(-1)[0]
        ),
        "restricted_logit": float(logit.numpy().reshape(-1)[0]),
        "gradcam_method": method,
        "heatmap_std": float(np.std(heatmap)),
    }


def resize_heatmap(heatmap):
    resized = tf.image.resize(
        heatmap[..., None], IMAGE_SIZE, method="bilinear"
    )
    return tf.squeeze(resized, axis=-1).numpy()


def save_three_panel(image_array, heatmap, save_path, title):
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.6))

    axes[0].imshow(image_array.astype("uint8"))
    axes[0].set_title("Original", fontsize=10)
    axes[0].axis("off")

    heatmap_artist = axes[1].imshow(
        heatmap, cmap="jet", vmin=0, vmax=1
    )
    axes[1].set_title("Grad-CAM heatmap", fontsize=10)
    axes[1].axis("off")

    axes[2].imshow(image_array.astype("uint8"))
    axes[2].imshow(
        heatmap, cmap="jet", alpha=OVERLAY_ALPHA, vmin=0, vmax=1
    )
    axes[2].set_title("Merged overlay", fontsize=10)
    axes[2].axis("off")

    fig.suptitle(title, fontsize=11)
    fig.colorbar(
        heatmap_artist,
        ax=axes,
        fraction=0.025,
        pad=0.02,
        label="Relative activation",
    )
    fig.subplots_adjust(
        left=0.02, right=0.95, bottom=0.03, top=0.86, wspace=0.08
    )
    fig.savefig(
        save_path, dpi=150, bbox_inches="tight", facecolor="white"
    )
    plt.close(fig)

    if not wait_for_file(save_path, min_bytes=1000):
        raise RuntimeError(f"Image was not written correctly: {save_path}")


In [ ]:
# 8. Reproduce the established confusion matrix
calibration_rows = []
selected_mode = None
selected_match = None
selected_probabilities = None

for mode in PREPROCESSING_MODES:
    print(f"Testing preprocessing: {mode}")
    dataset = make_prediction_dataset(all_test_images, mode)
    probabilities = model.predict(dataset, verbose=1).reshape(-1)

    mode_csv = OUTPUT_ROOT / f"predictions_{mode}.csv"
    pd.DataFrame({
        "image_path": [str(path) for path in all_test_images],
        "true_class": true_classes,
        "model_output": probabilities,
    }).to_csv(mode_csv, index=False)

    matches, best = find_exact_operating_point(probabilities)
    calibration_rows.append({
        "mode": mode,
        "exact_matches": len(matches),
        **best,
    })

    # Save diagnostics immediately, even if this mode is not correct.
    pd.DataFrame(calibration_rows).to_csv(
        OUTPUT_ROOT / "calibration_progress.csv", index=False
    )

    print("Best counts:", {
        key: best[key] for key in ["TP", "TN", "FP", "FN"]
    })
    print("Distance from expected:", best["distance"])

    if matches:
        # Prefer the operating point nearest 0.5 if several are equivalent.
        selected_match = min(
            matches, key=lambda row: abs(row["threshold"] - 0.5)
        )
        selected_mode = mode
        selected_probabilities = probabilities
        break

calibration_df = pd.DataFrame(calibration_rows)
display(calibration_df)

if selected_mode is None:
    raise RuntimeError(
        "No preprocessing/threshold/polarity combination reproduced "
        f"the required counts {EXPECTED_COUNTS}. Diagnostic CSV files "
        f"have been saved in {OUTPUT_ROOT}."
    )

PREPROCESSING_MODE = selected_mode
OPERATING_THRESHOLD = selected_match["threshold"]
POSITIVE_WHEN_HIGH = selected_match["positive_when_high"]

print("VERIFIED preprocessing:", PREPROCESSING_MODE)
print("VERIFIED threshold:", OPERATING_THRESHOLD)
print("Restricted class when output is high:", POSITIVE_WHEN_HIGH)
print("VERIFIED counts:", {
    key: selected_match[key] for key in ["TP", "TN", "FP", "FN"]
})


Testing preprocessing: raw_0_255
33/33 ━━━━━━━━━━━━━━━━━━━━ 543s 16s/step
Best counts: {'TP': 183, 'TN': 824, 'FP': 14, 'FN': 7}
Distance from expected: 0


,mode,exact_matches,threshold,positive_when_high,distance,TP,TN,FP,FN
0,raw_0_255,2,0.498298,True,0,183,824,14,7


VERIFIED preprocessing: raw_0_255
VERIFIED threshold: 0.5
Restricted class when output is high: True
VERIFIED counts: {'TP': 183, 'TN': 824, 'FP': 14, 'FN': 7}


In [ ]:
# 9. Build the exact review set and generate the images safely
prediction_rows = []

for image_path, true_class, probability in zip(
    all_test_images, true_classes, selected_probabilities
):
    predicted_class = classify_probability(
        float(probability),
        OPERATING_THRESHOLD,
        POSITIVE_WHEN_HIGH,
    )
    code = category_code(true_class, predicted_class)

    breed_label = image_path.stem.rsplit("_n", 1)[0].replace("_", " ")

    prediction_rows.append({
        "image_path": str(image_path),
        "original_filename": image_path.name,
        "breed_label": breed_label,
        "true_class": true_class,
        "predicted_class": predicted_class,
        "category_code": code,
        "model_output": float(probability),
    })

predictions_df = pd.DataFrame(prediction_rows)
predictions_df.to_csv(
    OUTPUT_ROOT / "verified_all_test_predictions.csv", index=False
)

full_counts = (
    predictions_df["category_code"]
    .value_counts()
    .reindex(["TP", "TN", "FP", "FN"], fill_value=0)
    .to_dict()
)

if full_counts != EXPECTED_COUNTS:
    raise RuntimeError(
        f"Verified counts changed unexpectedly: {full_counts}"
    )

selected_parts = []
for code in ["FP", "FN", "TN", "TP"]:
    subset = predictions_df[
        predictions_df["category_code"] == code
    ].copy()
    required = SAVE_COUNTS[code]

    if len(subset) < required:
        raise RuntimeError(
            f"Need {required} {code} images, but found {len(subset)}."
        )

    selected = subset.sample(
        n=required, random_state=SEED
    )

    selected_parts.append(selected)

selected_df = pd.concat(selected_parts, ignore_index=True)
selected_df.to_csv(
    OUTPUT_ROOT / "selected_12_images.csv", index=False
)

selected_counts = (
    selected_df["category_code"]
    .value_counts()
    .reindex(["TP", "TN", "FP", "FN"], fill_value=0)
    .to_dict()
)
log_status(f"Selected figures: {selected_counts}")

if selected_counts != SAVE_COUNTS:
    raise RuntimeError(
        f"Selection mismatch: expected {SAVE_COUNTS}, "
        f"got {selected_counts}"
    )

progress_path = OUTPUT_ROOT / "generation_progress.csv"
failure_path = OUTPUT_ROOT / "generation_failures.csv"

progress_columns = [
    "image_path", "original_filename", "breed_label", "true_class",
    "predicted_class", "category_code", "model_output",
    "saved_figure", "saved_bytes",
    "single_image_model_output", "restricted_logit",
    "gradcam_method", "heatmap_std",
]
failure_columns = [
    "image_path", "original_filename", "breed_label", "true_class",
    "predicted_class", "category_code", "error",
]

pd.DataFrame(columns=progress_columns).to_csv(progress_path, index=False)
pd.DataFrame(columns=failure_columns).to_csv(failure_path, index=False)

result_rows = []
failure_rows = []
start_time = time.time()

# Smoke-test the first selected image and confirm it really lands on Drive.
first_row = selected_df.iloc[0]
log_status(
    "Smoke test on first selected image: "
    f"{first_row['original_filename']}"
)

def process_one(row_dict):
    image_path = Path(row_dict["image_path"])
    image_array = load_rgb(image_path)
    image_batch = tf.expand_dims(
        preprocess_tensor(image_array, PREPROCESSING_MODE),
        axis=0,
    )

    heatmap, info = predict_and_explain(
        image_batch, row_dict["predicted_class"]
    )
    resized_heatmap = resize_heatmap(heatmap)

    if float(np.std(resized_heatmap)) <= FLAT_TOLERANCE:
        raise RuntimeError("Resized heatmap is flat.")

    code = row_dict["category_code"]
    save_path = (
        FIGURES_DIR
        / f"{code}_{safe_stem(image_path)}_gradcam.png"
    )

    title = (
        f"{code} | Breed: {row_dict['breed_label']} | "
        f"True: {row_dict['true_class']} | "
        f"Predicted: {row_dict['predicted_class']} | "
        f"Model output={row_dict['model_output']:.4f}"
    )
    save_three_panel(
        image_array, resized_heatmap, save_path, title
    )

    row_out = {
        **row_dict,
        "saved_figure": str(save_path),
        "saved_bytes": int(save_path.stat().st_size),
        "single_image_model_output": info["single_image_model_output"],
        "restricted_logit": info["restricted_logit"],
        "gradcam_method": info["gradcam_method"],
        "heatmap_std": info["heatmap_std"],
    }

    del image_array, image_batch, heatmap, resized_heatmap
    gc.collect()
    return row_out

for number, row in enumerate(
    selected_df.to_dict("records"), start=1
):
    try:
        row_out = process_one(row)
        result_rows.append(row_out)
        pd.DataFrame(result_rows).to_csv(progress_path, index=False)

        elapsed = time.time() - start_time
        remaining = elapsed / number * (len(selected_df) - number)

        log_status(
            f"{number}/{len(selected_df)} saved | "
            f"{row_out['category_code']} | "
            f"{Path(row_out['saved_figure']).name} | "
            f"Remaining≈{remaining/60:.1f} min"
        )

    except Exception as error:
        failure_row = {
            "image_path": row["image_path"],
            "original_filename": row["original_filename"],
            "breed_label": row["breed_label"],
            "true_class": row["true_class"],
            "predicted_class": row["predicted_class"],
            "category_code": row["category_code"],
            "error": f"{type(error).__name__}: {error}",
        }
        failure_rows.append(failure_row)
        pd.DataFrame(failure_rows).to_csv(failure_path, index=False)
        log_status(
            f"FAILED on {row['original_filename']} | "
            f"{failure_row['error']}"
        )
        raise

print("Image generation completed.")
log_status(
    f"Finished generating {len(result_rows)} figures."
)


[2026-07-18 20:26:00] Selected figures: {'TP': 3, 'TN': 3, 'FP': 3, 'FN': 3}
[2026-07-18 20:26:00] Smoke test on first selected image: black-and-tan_coonhound_n02089078_1454.jpg
[2026-07-18 20:26:07] 1/12 saved | FP | FP_black-and-tan_coonhound_n02089078_1454_gradcam.png | Remaining≈1.2 min
[2026-07-18 20:26:11] 2/12 saved | FP | FP_kelpie_n02105412_7370_gradcam.png | Remaining≈0.9 min
[2026-07-18 20:26:16] 3/12 saved | FP | FP_weimaraner_n02092339_3028_gradcam.png | Remaining≈0.8 min
[2026-07-18 20:26:22] 4/12 saved | FN | FN_staffordshire_bullterrier_n02093256_5325_gradcam.png | Remaining≈0.7 min
[2026-07-18 20:26:27] 5/12 saved | FN | FN_staffordshire_bullterrier_n02093256_5988_gradcam.png | Remaining≈0.6 min
[2026-07-18 20:26:31] 6/12 saved | FN | FN_staffordshire_bullterrier_n02093256_2987_gradcam.png | Remaining≈0.5 min
[2026-07-18 20:26:37] 7/12 saved | TN | TN_greater_swiss_mountain_dog_n02107574_2662_gradcam.png | Remaining≈0.4 min
[2026-07-18 20:26:42] 8/12 saved | TN | TN_ke

In [ ]:
# 10. Verify every required file and count
results_df = pd.read_csv(
    OUTPUT_ROOT / "generation_progress.csv"
)
failures_df = pd.read_csv(
    OUTPUT_ROOT / "generation_failures.csv"
)

all_files = list(FIGURES_DIR.glob("*.png"))
verified_counts = {
    code: len(list(FIGURES_DIR.glob(f"{code}_*.png")))
    for code in ["TP", "TN", "FP", "FN"]
}

for file_path in all_files:
    if file_path.stat().st_size < 1000:
        raise RuntimeError(
            f"Invalid image file: {file_path}"
        )

print("Full confusion matrix:", full_counts)
print("Saved image counts:", verified_counts)

if verified_counts != SAVE_COUNTS:
    raise RuntimeError(
        f"Saved-file mismatch: expected {SAVE_COUNTS}, "
        f"found {verified_counts}"
    )

if len(results_df) != sum(SAVE_COUNTS.values()):
    raise RuntimeError(
        f"Expected {sum(SAVE_COUNTS.values())} result rows, "
        f"found {len(results_df)}."
    )

if not failures_df.empty:
    raise RuntimeError(
        f"Failures were recorded: {len(failures_df)}"
    )

if (results_df["heatmap_std"] <= FLAT_TOLERANCE).any():
    raise RuntimeError("At least one saved Grad-CAM map is flat.")

RUN_COMPLETED_FILE.write_text(
    "Counts verified: TP=183, TN=824, FP=14, FN=7\n"
    "Images saved: TP=3, TN=3, FP=3, FN=3\n"
    "Dataset basis: 6 restricted breeds and the fixed 24-breed "
    "unrestricted sample produced by Notebook 01.\n",
    encoding="utf-8",
)

log_status("RUN COMPLETED AND VERIFIED")
print("Output folder:", OUTPUT_ROOT)
print("Figures folder:", FIGURES_DIR)


Full confusion matrix: {'TP': 183, 'TN': 824, 'FP': 14, 'FN': 7}
Saved image counts: {'TP': 3, 'TN': 3, 'FP': 3, 'FN': 3}
[2026-07-18 20:27:02] RUN COMPLETED AND VERIFIED
Output folder: /content/drive/MyDrive/restricted-dog-cnn/Explanation-v10
Figures folder: /content/drive/MyDrive/restricted-dog-cnn/Explanation-v10/figures


## Expected output

The `model_explainability` folder contains one `figures` folder with 12 PNG files: three TP, three TN, three FP, and three FN. Each filename begins with its category code, and the original breed-based filename is retained. CSV files and run-status files are saved directly in `model_explainability`.
